# NusaSynth - Metrik Single-LLM (baseline ablation)

Notebook TERPISAH dari `bab_iv_metrics.ipynb`. Beda hanya loader (`syn_dfs` -> output Single-LLM). Bagian pipeline outcome & conflict TIDAK ADA (Single-LLM tanpa mesin/validator).

## Setup

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 200)

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')
sys.path.insert(0, str(ROOT))
from NusaSynth.config import single_llm_synthetic_dir, synthetic_dir

LANGS = ['ace', 'ban', 'bjn', 'jav', 'mad', 'min', 'sun']
SENTENCES_PER_SEED = 5
BATCH_SIZE = 5

seed_dfs, syn_dfs, valid_dfs, test_dfs = {}, {}, {}, {}
for lang in LANGS:
    seed_p = ROOT / f'data/nusax_senti/{lang}/train.csv'
    syn_p = single_llm_synthetic_dir(lang) / 'synthetic.csv'
    valid_p = ROOT / f'data/nusax_senti/{lang}/valid.csv'
    test_p = ROOT / f'data/nusax_senti/{lang}/test.csv'
    if not (seed_p.exists() and syn_p.exists()):
        print(f'  [{lang}] SKIP: file tidak lengkap ({syn_p})')
        continue
    seed_dfs[lang] = pd.read_csv(seed_p)
    syn_dfs[lang] = pd.read_csv(syn_p)
    if valid_p.exists(): valid_dfs[lang] = pd.read_csv(valid_p)
    if test_p.exists(): test_dfs[lang] = pd.read_csv(test_p)
    print(f'  [{lang}] seed={len(seed_dfs[lang])} syn={len(syn_dfs[lang])}')
print('Sumber synthetic = SINGLE-LLM ->', single_llm_synthetic_dir(LANGS[0]).parent)

  [ace] seed=500 syn=2500
  [ban] seed=500 syn=2500
  [bjn] seed=500 syn=2500
  [jav] seed=500 syn=2500
  [mad] seed=500 syn=2500
  [min] seed=500 syn=2500
  [sun] seed=500 syn=2500
Sumber synthetic = SINGLE-LLM -> Y:\Michh\Python\Projects\MAGenerator\outputs\synthetic_single_llm\2026-07-23_22-12-08


## B. Keberagaman & Kebaruan - Self-BLEU (pers. 3.11) & BLEU vs Seed (pers. 3.12)
Fungsi identik `bab_iv_metrics.ipynb` (disalin verbatim).

In [2]:
# B2 — Self-BLEU (pers. 3.11) & BLEU vs Seed (pers. 3.12)
# BLEU = rata-rata geometrik presisi n-gram termodifikasi, bobot seragam 1/N (Papineni et al. 2002); 4 orde kumulatif 2-5, BLEU-4 acuan.
# Implementasi ikut Texygen SelfBleu.py: bobot seragam + method1 smoothing (eps=0.1).
#
# BEST PRACTICE: Self-BLEU sensitif ukuran |D| (makin banyak referensi -> makin tinggi). Data yg sama:
#   |D|~900 -> ~0.22 ; |D|~190 -> ~0.11. Jadi Self-BLEU sintetis dihitung pada |D| DISETARAKAN ke seed,
#   lalu dibandingkan ke Self-BLEU seed (data manusia) sbg baseline. Sampel acak deterministik seed=42.
smoothie = SmoothingFunction().method1
ORDERS = (2, 3, 4, 5)
WEIGHTS = [tuple([1.0 / n] * n) for n in ORDERS]  # kumulatif orde 1..n; list-of-weights -> satu call semua orde
SAMPLE_SEED = 42


def tokenize(text):
    return text.lower().split()


def _bleu_orders(refs_tokens, hyp_tokens):
    # satu call sentence_bleu -> semua orde; BLEU-4 identik bit-per-bit dgn (0.25,)*4 tunggal
    return dict(zip(ORDERS, sentence_bleu(refs_tokens, hyp_tokens, WEIGHTS, smoothing_function=smoothie)))


def self_bleu(sentences):
    # pers. 3.11 pada himpunan D: tiap kalimat vs semua lainnya (label sama), dirata-rata
    if len(sentences) < 2:
        return {n: 0.0 for n in ORDERS}
    T = [tokenize(s) for s in sentences]
    acc = {n: [] for n in ORDERS}
    for i in range(len(T)):
        d = _bleu_orders([T[j] for j in range(len(T)) if j != i], T[i])
        for n in ORDERS:
            acc[n].append(d[n])
    return {n: float(np.mean(acc[n])) for n in ORDERS}


def bleu_vs_ref(hyps, refs):
    # pers. 3.12: tiap kalimat sintetis vs seluruh korpus seed label sama, dirata-rata
    if not hyps or not refs:
        return {n: 0.0 for n in ORDERS}
    R = [tokenize(r) for r in refs]
    acc = {n: [] for n in ORDERS}
    for h in hyps:
        d = _bleu_orders(R, tokenize(h))
        for n in ORDERS:
            acc[n].append(d[n])
    return {n: float(np.mean(acc[n])) for n in ORDERS}


def match_sample(items, n):
    # sampel acak deterministik ukuran n (menyetarakan |D| sintetis ke seed)
    if len(items) <= n:
        return list(items)
    rng = np.random.default_rng(SAMPLE_SEED)
    return [items[i] for i in rng.choice(len(items), n, replace=False)]

In [3]:
REF_ORDER = 4  # orde acuan (konfigurasi standar N=4, Papineni et al. 2002)
METRICS = [('self_syn', 'Self-BLEU sintetis'), ('self_seed', 'Self-BLEU seed'), ('bs', 'BLEU vs Seed')]

rows_head, rows_multi = [], []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang], syn_dfs[lang]
    pl = {}
    for label in ['negative', 'neutral', 'positive']:
        seed_t = seed_df[seed_df['label'] == label]['text'].dropna().tolist()
        syn_all = syn_df[syn_df['label'] == label]['text'].dropna().tolist()
        if len(syn_all) < 2 or len(seed_t) < 2:
            continue
        syn_m = match_sample(syn_all, len(seed_t))       # Self-BLEU: |D| sintetis = |D| seed (adil, pers. 3.11)
        pl[label] = {
            'self_syn':  self_bleu(syn_m),               # keberagaman sintetis (matched ke seed)
            'self_seed': self_bleu(seed_t),              # keberagaman seed (baseline manusia)
            'bs':        bleu_vs_ref(syn_all, seed_t),   # BLEU-vs-Seed: SELURUH sintetis vs seed (pers. 3.12)
        }
    if not pl:
        continue
    macro = lambda k, n: float(np.mean([pl[l][k][n] for l in pl]))

    # Headline BLEU-4 (orde acuan) — ketiga metrik berdampingan
    rows_head.append({
        'lang': lang,
        'selfBLEU_syn':  round(macro('self_syn',  REF_ORDER), 4),
        'selfBLEU_seed': round(macro('self_seed', REF_ORDER), 4),
        'bleu_vs_seed':  round(macro('bs',        REF_ORDER), 4),
    })
    # Multi-orde BLEU-2/3/4/5 — ketiga metrik (selaras Zhu 2018, Tabel 2 & 3)
    for key, name in METRICS:
        rows_multi.append({'lang': lang, 'metrik': name,
                           **{f'BLEU-{n}': round(macro(key, n), 4) for n in ORDERS}})

print(f'HEADLINE BLEU-{REF_ORDER} (orde acuan) | Self-BLEU: |D| disamakan ke seed (~190/label, seed=42) | BLEU-vs-Seed: seluruh sintetis (pers. 3.12)')
display(pd.DataFrame(rows_head))
print('\nMULTI-ORDE BLEU-2/3/4/5 (selaras Zhu 2018) — orde tinggi lebih peka pengulangan struktur panjang')
display(pd.DataFrame(rows_multi))

HEADLINE BLEU-4 (orde acuan) | Self-BLEU: |D| disamakan ke seed (~190/label, seed=42) | BLEU-vs-Seed: seluruh sintetis (pers. 3.12)


,lang,selfBLEU_syn,selfBLEU_seed,bleu_vs_seed
0,ace,0.1809,0.0554,0.1785
1,ban,0.1043,0.0432,0.0997
2,bjn,0.1153,0.0545,0.1034
3,jav,0.0856,0.0502,0.0664
4,mad,0.1979,0.0606,0.1438
5,min,0.0960,0.0495,0.0969
6,sun,0.1065,0.0516,0.0892



MULTI-ORDE BLEU-2/3/4/5 (selaras Zhu 2018) — orde tinggi lebih peka pengulangan struktur panjang


,lang,metrik,BLEU-2,BLEU-3,BLEU-4,BLEU-5
0,ace,Self-BLEU sintetis,0.4186,0.2508,0.1809,0.1450
1,ace,Self-BLEU seed,0.2754,0.1070,0.0554,0.0370
2,ace,BLEU vs Seed,0.4138,0.2552,0.1785,0.1381
3,ban,Self-BLEU sintetis,0.3383,0.1685,0.1043,0.0750
4,ban,Self-BLEU seed,0.2162,0.0761,0.0432,0.0318
5,ban,BLEU vs Seed,0.2931,0.1540,0.0997,0.0734
6,bjn,Self-BLEU sintetis,0.3774,0.1991,0.1153,0.0755
7,bjn,Self-BLEU seed,0.2685,0.1075,0.0545,0.0371
8,bjn,BLEU vs Seed,0.3397,0.1717,0.1034,0.0733
9,jav,Self-BLEU sintetis,0.3511,0.1592,0.0856,0.0553


## C. Kesesuaian Distribusi Panjang - R_sigma (3.13) & Delta_mu (3.14)
Single-LLM tanpa PSLM -> R_sigma diperkirakan jauh dari 1 (verdict 'CHECK' wajar, itu kontrafaktualnya).

In [4]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang].copy(), syn_dfs[lang].copy()
    seed_df['wc'] = seed_df['text'].str.split().str.len()
    syn_df['wc']  = syn_df['text'].str.split().str.len()
    for label in ['negative', 'neutral', 'positive']:
        s_seed = seed_df[seed_df['label'] == label]['wc']
        s_syn  = syn_df[syn_df['label'] == label]['wc']
        if len(s_seed) > 1 and len(s_syn) > 1:
            ratio = round(s_syn.std() / s_seed.std(), 2)
            mean_diff = round(s_syn.mean() - s_seed.mean(), 1)
            verdict = 'OK' if 0.80 <= ratio <= 1.20 and abs(mean_diff) <= 5 else 'CHECK'
            rows.append({
                'lang': lang,
                'label': label,
                'seed_n': len(s_seed),
                'syn_n': len(s_syn),
                'seed_mean': round(s_seed.mean(), 1),
                'syn_mean': round(s_syn.mean(), 1),
                'mean_diff': mean_diff,
                'seed_std': round(s_seed.std(), 1),
                'syn_std': round(s_syn.std(), 1),
                'std_ratio': ratio,
                'verdict': verdict,
            })

display(pd.DataFrame(rows))

,lang,label,seed_n,syn_n,seed_mean,syn_mean,mean_diff,seed_std,syn_std,std_ratio,verdict
0,ace,negative,192,960,22.5,20.1,-2.4,15.0,10.5,0.70,CHECK
1,ace,neutral,119,595,14.5,14.7,0.2,9.6,8.4,0.87,OK
2,ace,positive,189,945,31.2,27.3,-3.9,15.4,11.4,0.74,CHECK
3,ban,negative,192,960,21.9,19.0,-3.0,13.8,9.3,0.67,CHECK
4,ban,neutral,119,595,13.8,14.3,0.4,8.5,7.3,0.85,OK
5,ban,positive,189,945,30.8,25.4,-5.4,14.4,9.5,0.66,CHECK
6,bjn,negative,192,960,21.9,21.1,-0.8,14.1,10.4,0.74,CHECK
7,bjn,neutral,119,595,13.5,14.1,0.6,8.3,7.5,0.90,OK
8,bjn,positive,189,945,30.6,26.7,-3.8,14.4,10.3,0.72,CHECK
9,jav,negative,192,960,21.8,20.2,-1.6,14.2,9.7,0.68,CHECK


In [5]:
def len_stats(df):
    df = df.copy()
    df['wc'] = df['text'].str.split().str.len()
    return {
        'n': len(df),
        'mean': round(df['wc'].mean(), 1),
        'std': round(df['wc'].std(), 1),
        'min': int(df['wc'].min()),
        'max': int(df['wc'].max()),
        'median': int(df['wc'].median()),
        'p95': int(df['wc'].quantile(0.95)),
    }

rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    rows.append({'lang': lang, 'split': 'seed', **len_stats(seed_dfs[lang])})
    rows.append({'lang': lang, 'split': 'syn',  **len_stats(syn_dfs[lang])})
    if lang in valid_dfs:
        rows.append({'lang': lang, 'split': 'valid', **len_stats(valid_dfs[lang])})
    if lang in test_dfs:
        rows.append({'lang': lang, 'split': 'test',  **len_stats(test_dfs[lang])})

display(pd.DataFrame(rows))

,lang,split,n,mean,std,min,max,median,p95
0,ace,seed,500,23.9,15.4,4,94,20,56
1,ace,syn,2500,21.5,11.5,2,68,20,44
2,ace,valid,100,23.4,13.9,4,68,20,51
3,ace,test,400,24.1,15.2,4,75,20,57
4,ban,seed,500,23.4,14.6,3,71,19,53
5,ban,syn,2500,20.3,10.0,3,61,19,39
6,ban,valid,100,23.4,13.5,4,64,21,48
7,ban,test,400,23.5,14.5,5,68,20,57
8,bjn,seed,500,23.2,14.6,4,77,19,53
9,bjn,syn,2500,21.6,10.9,4,66,20,44


## D. Audit Classifier (GlotLID purity, NusaBERT label-drift, duplication)

In [6]:
run_dir = single_llm_synthetic_dir(LANGS[0]).parent
audit_path = run_dir / 'audit_summary.json'
if audit_path.exists():
    audit = pd.DataFrame(json.loads(audit_path.read_text(encoding='utf-8')))
    print('Audit classifier Single-LLM (dari scripts/single_llm_audit.py):')
    display(audit)
    print('\nmean 7 bahasa: dup={:.2f}%  lang_confusion={:.2f}%  label_drift={:.2f}%'.format(
        audit['dup_rate'].mean(), audit['lang_confusion'].mean(), audit['label_drift'].mean()))
else:
    print(f'Belum ada {audit_path} -> jalankan dulu: uv run python scripts/single_llm_audit.py')

Audit classifier Single-LLM (dari scripts/single_llm_audit.py):


,lang,n,dup_rate,lang_confusion,label_drift
0,jav,2500,0.52,6.96,7.52
1,sun,2500,1.72,3.20,8.72
2,ace,2500,13.32,8.16,13.16
3,bjn,2500,1.92,12.16,6.16
4,mad,2500,15.00,12.72,12.04
5,min,2500,1.72,5.72,14.04
6,ban,2500,4.12,17.60,9.72



mean 7 bahasa: dup=5.47%  lang_confusion=9.50%  label_drift=10.19%


## E. Perbandingan vs NusaSynth Penuh

In [7]:
def mu_selfbleu4(get_dir):
    vals = []
    for lang in LANGS:
        seed_df = seed_dfs.get(lang)
        if seed_df is None:
            continue
        p = get_dir(lang) / 'synthetic.csv'
        if not p.exists():
            continue
        syn_df = pd.read_csv(p)
        per = []
        for label in ['negative', 'neutral', 'positive']:
            seed_t = seed_df[seed_df['label'] == label]['text'].dropna().tolist()
            syn_all = syn_df[syn_df['label'] == label]['text'].dropna().tolist()
            if len(syn_all) < 2 or len(seed_t) < 2:
                continue
            per.append(self_bleu(match_sample(syn_all, len(seed_t)))[4])
        if per:
            vals.append(float(np.mean(per)))
    return round(float(np.mean(vals)), 4) if vals else None

print('Self-BLEU-4 mu (7 bahasa) - lebih RENDAH = lebih beragam:')
print('  Single-LLM      :', mu_selfbleu4(single_llm_synthetic_dir))
print('  NusaSynth penuh :', mu_selfbleu4(synthetic_dir))

Self-BLEU-4 mu (7 bahasa) - lebih RENDAH = lebih beragam:


  Single-LLM      : 0.1267


  NusaSynth penuh : 0.1056


In [8]:
# ===== RINGKASAN AGREGAT Single-LLM — angka yang dilaporkan di Tabel 4.9 (perbandingan vs pipeline penuh) =====
print('=' * 60); print('AGREGAT Single-LLM (angka Tabel 4.9)'); print('=' * 60)

# 1. GlotLID Identification Rate = 100 - lang_confusion (dari audit_summary.json)
run_dir = single_llm_synthetic_dir(LANGS[0]).parent
audit = pd.DataFrame(json.loads((run_dir / 'audit_summary.json').read_text(encoding='utf-8')))
print(f"\n1. GlotLID Identification Rate = {100 - audit['lang_confusion'].mean():.2f}%  "
      f"(= 100 - lang_confusion {audit['lang_confusion'].mean():.2f}%)")

# 2. Self-BLEU & BLEU vs Seed (mean 7 bahasa, BLEU-4) dari rows_head (sel B)
h = pd.DataFrame(rows_head)
print(f"2. Self-BLEU sintetis = {h['selfBLEU_syn'].mean():.4f}  Self-BLEU seed = {h['selfBLEU_seed'].mean():.4f}  "
      f"BLEU vs Seed = {h['bleu_vs_seed'].mean():.4f}")

# 3. Rasio Std Panjang (R_sigma) & Selisih Rata Panjang (Delta_mu) — recompute dari data single-LLM
rr = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    sd = seed_dfs[lang].copy(); yd = syn_dfs[lang].copy()
    sd['wc'] = sd['text'].str.split().str.len(); yd['wc'] = yd['text'].str.split().str.len()
    for lab in ['negative', 'neutral', 'positive']:
        a = sd[sd['label'] == lab]['wc']; b = yd[yd['label'] == lab]['wc']
        if len(a) > 1 and len(b) > 1:
            rr.append({'R': b.std() / a.std(), 'D': b.mean() - a.mean()})
rd = pd.DataFrame(rr)
print(f"3. R_sigma rata-rata = {rd['R'].mean():.2f}  Delta_mu rata-rata = {rd['D'].mean():+.2f} kata")
print("\n(Bandingkan pipeline penuh di bab_iv_metrics.ipynb: GlotLID 97,72 | Self-BLEU 0,1056 | BLEU-vs-Seed 0,0755 | R_sigma 0,79 | Delta_mu +0,53)")


AGREGAT Single-LLM (angka Tabel 4.9)

1. GlotLID Identification Rate = 90.50%  (= 100 - lang_confusion 9.50%)
2. Self-BLEU sintetis = 0.1266  Self-BLEU seed = 0.0521  BLEU vs Seed = 0.1111
3. R_sigma rata-rata = 0.78  Delta_mu rata-rata = -1.82 kata

(Bandingkan pipeline penuh di bab_iv_metrics.ipynb: GlotLID 97,72 | Self-BLEU 0,1056 | BLEU-vs-Seed 0,0755 | R_sigma 0,79 | Delta_mu +0,53)
